<br/>

<img src="https://uploads-ssl.webflow.com/614b1fe22fa8b90ef41aeffe/6265cb48f9496b1cefc9ab75_logotipo-mbit-39.png" width="200px" align="right" CLASS="TextWrap" style="background-color:#2a3f3f;">

<h1><font color="#2a3f3f" size=5>Generación y procesamiento avanzado de listas</font></h1>

<br/>
<div style="text-align: right">
<font color="#2a3f3f" size=3>Javier Cózar - javier.cozar@mbitschool.com</font><br>
<font color="#2a3f3f" size=3>Python avanzado</font><br>
</div>


## Eficiencia sobre una lista de elementos

Cuando trabajemos con conjuntos de datos, sobre todo cuando éstos son especialmente grandes, es muy importante tener en cuenta la eficiencia.

En esta sección abordaremos detalles de implementación que tienen un gran impacto en el rendimiento de nuestro código.

### Inicialización de listas

Como sabemos, no es necesario especificar el tamaño de las listas en su creación (al contrario que en otros lenguajes de proogramación). Esto, aunque muy útil, conlleva una penalización en cuanto al uso de memoria RAM. Además, exite un sobrecoste en tiempo de ejecución cuando hay que expandir el tamaño de la lista (uso de la función `append`).

In [1]:
%%timeit -n 10
# Uso de append (va aumentando el tamaño de la lista dinámicamente)

l = []
N = 10**6
for i in range(N):
    l.append(i)

40 ms ± 809 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [2]:
%%timeit -n 10
# Inicializamos la lista a un tamaño N (usando elementos None): es más rápido

N = 10**6
l = [0] * N
for i in range(N):
    l[i] = i

31.7 ms ± 1.34 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Iteradores y generadores: ¿Qué son?

Conceptualmente, un iterador es cualquier objeto que se puede iterar, es decir, recorrer sus elementos mediante un bucle o equivalente (acaba llamando a una función interna `__iter__`).

Un generador es un objeto que genera un elemento cada vez que se le solicita (a través de una llamada a una función interna `__next__`). Todo generador es un iterador, pero no todo iterador es un generador: por ejemplo, la función `enumerate` devuelve un generador (e iterador) pero la función `range` devuelve un iterador (pero no generador).

In [3]:
generator = enumerate(["A", "B", "C"])
# funciona
for i, x in generator:
    print(i, x)

0 A
1 B
2 C


In [4]:
generator = enumerate(["A", "B", "C"])
# funciona
next(generator)

(0, 'A')

In [9]:
iterator = range(5)
# funciona
for i in iterator:
    print(i)

0
1
2
3
4


In [11]:
iterator = range(5)
# falla
# next(iterator)

<div class="alert alert-block alert-info">
<i class="fa fa-info-circle" aria-hidden="true"></i>
Cuando consumimos todos los elementos de un generador usando next, si volvemos a usar next se lanzará una excepción.
</div>

In [14]:
generator = enumerate(["A", "B", "C"])
# funcionan
print( next(generator) )
print( next(generator) )
print( next(generator) )

# falla
# print( next(generator) )

(0, 'A')
(1, 'B')
(2, 'C')


La diferencia entre manejar un iterable y la propia lista de elementos radica en la eficiencia: mientras que la **lista de elementos debe permanecer alojada (por completo) en memoria principal** (habiéndose generado inicialmente por completo), el **iterable puede ir creando los objetos dinámicamente uno a uno**. Ejemplo usando `range`:

#### Range

In [15]:
%%timeit -n 10
N = 10 ** 6
suma = 0
for _ in list(range(N)):  # Creando la lista y luego iterándola
    suma += 1

37.8 ms ± 739 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
%%timeit  -n 10
N = 10 ** 6
suma = 0
for _ in range(N):  # Iterando el objeto range (iterable) directamente
    suma += 1

27.3 ms ± 153 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


Lo realmente costoso no está en el bucle (en ambos casos el bucle itera N veces la operación +=1). Lo costoso está en la creación de la lista entera en memoria, con el consecuente consumo de memoria para alojarla.

De hecho, la creación de un iterador o generador es inmediata puesto que no genera datos. Esto se puede observar en los tiempos de ejecución:

In [17]:
%%timeit
N = 10 ** 6
l = list(range(N))

12.7 ms ± 243 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [18]:
%%timeit
N = 10 ** 6
r = range(N)

94.6 ns ± 0.0614 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


## Comprehension lists

En los ejemplos anteriores hemos creado inicialmente una lista de resultados (suma), y le asignamos los valores en un bucle posterior. Esto significa que:

* Usamos append para ir añadiendo los valores, uno a uno, a la lista
* Gastamos tiempo en crear la lista en memoria (con valores None por ejemplo), y posteriormente asignamos los valores

Existe una alternativa más eficiente: los **comprehension lists**. La comprensión de listas o comprehension lists es una forma de generar una lista en una sola instrucción, especificando el contenido de la misma de una forma más compleja que con la función `range`. Sigue la misma sintaxis que un **bucle for**, pero combinado con la definición de una lista (**corchetes**), con la ventaja de que el código implícito está optimizado **(mucho más eficiente)**.

En este sentido, se pueden crear listas, generadores y diccionarios.

### Listas

A modo de ejemplo, la siguiente _comprehension list_ crea una lista del 0 al 10 (exclusive).

In [20]:
numeros = [i for i in range(10)]
numeros

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

In [22]:
# equivalente con bucle for
numeros = []

for i in range(10):
    numeros.append(i )
numeros

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]

Si nos fijamos bien, la sintáxis es similar al bucle for pero cambiando de orden los elementos:

1. En primer lugar se especifica la expresión que determina los valores dentro de la lista generada
2. En segundo lugar se especifica la definición del bucle for
3. Opcionalmente se puede especificar una operación de filtrado, usando `if` y una expresión booleana. Si la expresión se evalúa a `False` para un elemento, éste no será incluido en la lista.


Por ejemplo, a continuación vamos a generar una lista del 0 al 10 (exclusive) pero solo con los números pares.

In [23]:
numeros_pares = [i for i in range(10) if i % 2 == 0]
numeros_pares

[0, 2, 4, 6, 8]

In [24]:
# equivalente con bucle for
numeros_pares = []

for i in range(10):
    if i % 2 == 0:
        numeros_pares.append(i)
numeros_pares

[0, 2, 4, 6, 8]

Para terminar de ver la utilidad de este recurso, más allá de la eficiencia, vamos a describir el siguiente ejemplo.

Supongamos que tenemos dos listas, una que contiene el nombre de los jugadores de un equipo de baloncesto y otra contiene booleanos, indicando para cada jugador si va a ser titular o no en el próximo partido. Si geremos generar la lista de jugadores titulares sin usar comprehension lists tendremos que usar el siguiente código:

In [26]:
jugadores = ["Juan", "Pedro", "Mario", "Raúl", "Alberto", "José", "Manuel"]
titulares = [True, False, True, True, False, True, True]

In [27]:
jugadores_titulares = []
# asumimos que la longitud de ambas listas es la misma
for i in range(len(jugadores)):
    if titulares[i] == True:
        jugadores_titulares.append(jugadores[i])
jugadores_titulares

['Juan', 'Mario', 'Raúl', 'José', 'Manuel']

Para evitar tener que iterar sobre el índice y acceder a cada lista por separado usaremos la función `zip`:

In [28]:
jugadores_titulares = []
# asumimos que la longitud de ambas listas es la misma
for jugador, titular in zip(jugadores, titulares):
    if titular:
        jugadores_titulares.append(jugador)
jugadores_titulares

['Juan', 'Mario', 'Raúl', 'José', 'Manuel']

Usando `zip` hemos conseguido que el código sea más legible y más fácil de escribir. Vamos ahora a implementar el mismo código pero usando _comprehension lists_:

In [29]:
jugadores_titulares = [jugador for jugador, titular in zip(jugadores, titulares) if titular]
jugadores_titulares

['Juan', 'Mario', 'Raúl', 'José', 'Manuel']

Además de escribir mucho menos código es más eficiente (rápido). Podemos ganar aún más en legibilidad usando los saltos de línea. Personalmente, creo que se lee mejor estos comprehension lists si los implementamos en 3 lineas:

1. Primera parte, es decir, la expresión que determina los valores dentro de la lista generada
2. Segunda parte, es decir, la definición del bucle for
3. Opcionalmente la tercera parte, es decir, la condicional

In [30]:
jugadores_titulares = [
    jugador
    for jugador, titular in zip(jugadores, titulares)
    if titular
]
jugadores_titulares

['Juan', 'Mario', 'Raúl', 'José', 'Manuel']

### Expresiones generadoras

Esta sintáxis también se puede utilizar para crear generadores, es decir, como las comprehension lists pero sin generar toda la información en el momento sino que se pospone al momento de iterarla o consumirla. Para construirlas, basta con usar paréntesis en lugar de corchetes:

In [42]:
N = 10 ** 100
lista1 = range(0,N)
lista2 = range(0,N)

suma = ( 
    e1 + e2
    for e1, e2 in zip(lista1, lista2)
)

In [43]:
suma

<generator object <genexpr> at 0x110cd3f90>

In [53]:
next(suma)

18

In [39]:
len(list(suma))

999994

### Comprehension dict

Finalmente podemos usar esta misma sintaxis para construir diccionarios en lugar de secuencias de valores. Para ello, en lugar de expresar un único valor indicaremos un par clave-valor separado por `:`

In [54]:
N = 10 ** 6
lista1 = range(0,N)
lista2 = range(0,N)

suma = {
    (e1, e2): e1 + e2
    for e1, e2 in zip(lista1, lista2)
}

In [56]:
list(suma.keys())[:5]

[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4)]

In [57]:
suma[(2345, 2345)]

4690

## Programación funcional

Las funciones `filter`, `map` y `reduce` permiten aplicar una función sobre los elementos de un iterable. Las tres funciones tienen dos argumentos: el primero es la función a aplicar, y el segundo es el objeto iterable sobre el que aplicarlos.

* La función `filter` filtra los elementos del iterable en base al resultado una función que devuelve un booleano con cada elemento de la lista. Si es `True` permanece en el resultado, y si es `False` lo filtra.

* La función `map` permite aplicar una función a cada uno de los elementos, generando un segundo iterable.

* La función `reduce` permite combinar todos los elementos de la lista aplicando una finción, dos a dos iterativamente, hasta reducir a un solo elemento.

####  Filter

In [70]:
# Filtrar los numeros impares
l = range(10)
def es_par(x):
    return x % 2 == 0
pares = filter(es_par, l)
print(pares)
print(list(pares))

[0, 2, 4, 6, 8]


In [58]:
# ¡En este caso es mucho más cómodo usar funciones lambda!
l = range(10)
pares = filter(lambda x: x%2==0, l)
print(pares)
print(list(pares))

[0, 2, 4, 6, 8]


#### Map


In [75]:
l = range(10)
def cuadrado(x):
    return x**2
cuadrados = map(cuadrado, l)
print(cuadrados)
print(list(cuadrados))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


A veces es útil o cómodo emplear funciones lambda para este tipo de programación funcional:

In [59]:
l = range(10)

cuadrados = map(lambda x: x ** 2, l)
print(list(cuadrados))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


#### Reduce


Aunque en python 3.0 la función `reduce` se [eliminó](https://docs.python.org/3.0/whatsnew/3.0.html#builtins) como función `built-in` (y se movió a un módulo, llamado `functools`), es útil que la veamos porque será de utilizad en posteriores módulos (al menos conceptualmente).

La razón por la que la movieron, según la documentación oficial, es:

> _"Removed reduce(). Use functools.reduce() if you really need it; however, 99 percent of the time an explicit for loop is more readable."_

Sin embargo, aunque puede resultar más dificil de leer, requiere menos código (evitamos el uso de bucles), y desde el punto de vista de la _programación funcional_ es más **intuitiva**.

In [79]:
import functools
l = range(10)
suma = functools.reduce(lambda x, y: x+y, l)
print(suma)

45
